In [ ]:
!pip install libtorrent

In [ ]:
from google.colab import drive
import os, time, datetime, threading, logging
import libtorrent as lt

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive')
DOWNLOAD_FOLDER = '/content/drive/My Drive/Torrent'
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
logging.info(f"Download directory: {DOWNLOAD_FOLDER}")

# ── Session ───────────────────────────────────────────────────────────────────
def create_session() -> lt.session:
    settings = {
        'enable_dht':                True,
        'enable_lsd':                True,
        'enable_natpmp':             True,
        'enable_upnp':               True,
        'active_downloads':          5,
        'active_seeds':              5,
        'active_limit':              100,
        'connection_speed':          500,
        'download_rate_limit':       0,
        'upload_rate_limit':         10 * 1024 * 1024,
        'unchoke_slots_limit':       50,
        'aio_threads':               8,
        'file_pool_size':            500,
        'send_buffer_watermark':     500  * 1024,
        'send_buffer_low_watermark': 10   * 1024,
        'send_socket_buffer_size':   2    * 1024 * 1024,
        'recv_socket_buffer_size':   2    * 1024 * 1024,
        'max_peerlist_size':         50000,
        'dht_bootstrap_nodes':       'router.utorrent.com:6881,router.bittorrent.com:6881,dht.transmissionbt.com:6881',
    }
    ses = lt.session(settings)
    logging.info("Session created.")
    return ses

# ── Progress display ──────────────────────────────────────────────────────────
_STATE_STR = {
    0: 'Queued',
    1: 'Checking',
    2: 'Metadata',
    3: 'Downloading',
    4: 'Finished',
    5: 'Seeding',
    6: 'Allocating',
    7: 'Resuming',
}

def track_progress(handle: lt.torrent_handle):
    start = time.time()
    last  = 0.0

    while True:
        s = handle.status()
        if s.state == lt.torrent_status.seeding and s.progress >= 1.0:
            break
        now = time.time()
        if now - last >= 2:
            pct   = s.progress * 100
            dl    = s.download_rate / 1024
            ul    = s.upload_rate   / 1024
            peers = s.num_peers
            state = _STATE_STR.get(int(s.state), str(s.state))
            if s.download_rate > 0:
                secs_left = (1 - s.progress) * s.total_wanted / s.download_rate
                eta = str(datetime.timedelta(seconds=int(secs_left)))
            else:
                eta = '∞'
            print(
                f"\r{pct:6.2f}%  ↓ {dl:8.1f} kB/s  ↑ {ul:7.1f} kB/s  "
                f"Peers: {peers:3}  ETA: {eta:<12}  [{state}]   ",
                end='', flush=True
            )
            last = now
        time.sleep(0.5)

    elapsed = datetime.timedelta(seconds=int(time.time() - start))
    print(f"\n✅  Done in {elapsed}  →  {DOWNLOAD_FOLDER}")

# ── Piece prioritisation ──────────────────────────────────────────────────────
def prioritise_pieces(handle: lt.torrent_handle):
    info  = handle.torrent_file()
    total = info.num_pieces()
    edge  = max(1, int(total * 0.05))
    for p in range(edge):
        handle.piece_priority(p, 7)
    for p in range(total - edge, total):
        handle.piece_priority(p, 7)

# ── Size formatter ────────────────────────────────────────────────────────────
def fmt_size(b: int) -> str:
    if b >= 1024 ** 3:
        return f"{b / 1024**3:.2f} GB"
    elif b >= 1024 ** 2:
        return f"{b / 1024**2:.2f} MB"
    return f"{b / 1024:.2f} KB"

# ── Core download ─────────────────────────────────────────────────────────────
_ses    = None
_handle = None

def download(source: str):
    global _ses, _handle
    _ses = create_session()

    params = lt.add_torrent_params()
    params.save_path    = DOWNLOAD_FOLDER
    params.storage_mode = lt.storage_mode_t.storage_mode_sparse

    if source.startswith('magnet:'):
        params = lt.parse_magnet_uri(source)
        params.save_path    = DOWNLOAD_FOLDER
        params.storage_mode = lt.storage_mode_t.storage_mode_sparse
    elif os.path.isfile(source):
        params.ti = lt.torrent_info(source)
    else:
        raise ValueError(f"Not a magnet link or valid .torrent path: {source!r}")

    _handle = _ses.add_torrent(params)

    # ── Wait for metadata ─────────────────────────────────────────────────────
    METADATA_TIMEOUT = 120
    print("⏳ Fetching metadata", end='', flush=True)
    deadline = time.time() + METADATA_TIMEOUT
    while not _handle.status().has_metadata:
        if time.time() > deadline:
            raise TimeoutError(f"Metadata not received within {METADATA_TIMEOUT}s.")
        print('.', end='', flush=True)
        time.sleep(1)
    print(' done.')

    prioritise_pieces(_handle)
    info       = _handle.torrent_file()
    size_bytes = info.total_size()

    print(f"\n{'─'*58}")
    print(f"  📦  {info.name()}")
    print(f"  💾  Size   : {fmt_size(size_bytes)}  ({size_bytes:,} bytes)")
    print(f"  🧩  Pieces : {info.num_pieces()}  ×  {info.piece_length() // 1024} KB")
    print(f"  📁  Saving : {DOWNLOAD_FOLDER}")
    print(f"{'─'*58}\n")

    _handle.resume()
    track_progress(_handle)

def _cleanup():
    global _ses, _handle
    if _ses and _handle and _handle.is_valid():
        _ses.pause()
        _ses.remove_torrent(_handle)
    _ses = _handle = None

# ── Entry point ───────────────────────────────────────────────────────────────
source = input("Magnet link or .torrent file path: ").strip()

t = threading.Thread(target=download, args=(source,), daemon=True)
t.start()

try:
    while t.is_alive():
        time.sleep(1)
except KeyboardInterrupt:
    logging.warning("Interrupted — cleaning up…")
    _cleanup()